In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt

# 1) 하이퍼파라미터
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 64      # 메모리 여유를 위해 128→64로 줄였습니다
epochs = 20
img_size = 28
timesteps = 1000     # 훈련용 타임스텝

# 2) 노이즈 스케줄 (linear)
beta_start, beta_end = 1e-4, 0.02
betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
alphas = 1 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

# 3) 간단한 U-Net 스타일 모델
class SimpleUnet(nn.Module):
    def __init__(self, c=1, base=32):
        super().__init__()
        self.conv1 = nn.Conv2d(c, base, 3, 1, 1)
        self.conv2 = nn.Conv2d(base, base*2, 3, 2, 1)
        self.conv3 = nn.Conv2d(base*2, base*2, 3, 1, 1)
        self.up    = nn.Upsample(scale_factor=2, mode='nearest')
        self.conv4 = nn.Conv2d(base*2, base, 3, 1, 1)
        self.conv5 = nn.Conv2d(base, c, 3, 1, 1)

    def forward(self, x, t):
        emb = t.float()[:, None, None, None] / timesteps
        x = x + emb
        h1 = F.relu(self.conv1(x))
        h2 = F.relu(self.conv2(h1))
        h3 = F.relu(self.conv3(h2))
        h = self.up(h3)
        h = F.relu(self.conv4(h))
        return self.conv5(h)

model = SimpleUnet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# 4) 데이터 & transform
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1,1]
])
ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)

# 간단한 CUDA 체크 (한 번만)
print("CUDA available:", torch.cuda.is_available())
if device.startswith('cuda'):
    print("Device:", torch.cuda.get_device_name(0))

# 5) 순방향(noising) 함수
def q_sample(x0, t):
    noise = torch.randn_like(x0)
    a_t = alphas_cumprod[t][:, None, None, None]
    return torch.sqrt(a_t)*x0 + torch.sqrt(1-a_t)*noise, noise

# 6) 역방향(sampling) 함수
@torch.no_grad()
def p_sample(x_t, t):
    beta_t = betas[t][:, None, None, None]
    a_t = alphas[t][:, None, None, None]
    a_cum = alphas_cumprod[t][:, None, None, None]
    eps_pred = model(x_t, t)
    mean = (1/torch.sqrt(a_t)) * (x_t - beta_t/torch.sqrt(1-a_cum)*eps_pred)
    if t[0] > 0:
        noise = torch.randn_like(x_t)
        sigma = torch.sqrt(beta_t)
        return mean + sigma*noise
    else:
        return mean

def sample_loop(n):
    x = torch.randn(n, 1, img_size, img_size).to(device)
    for i in reversed(range(timesteps)):
        t = torch.full((n,), i, dtype=torch.long, device=device)
        x = p_sample(x, t)
    return (x.clamp(-1,1) + 1) / 2

# 7) 학습 루프 (샘플링 제거)
for epoch in range(1, epochs+1):
    model.train()
    total_loss = 0
    for x0, _ in dl:
        x0 = x0.to(device)
        bs = x0.size(0)
        t = torch.randint(0, timesteps, (bs,), device=device)
        x_t, noise = q_sample(x0, t)
        noise_pred = model(x_t, t)
        loss = F.mse_loss(noise_pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * bs

    print(f"Epoch {epoch}/{epochs} — Loss: {total_loss/len(ds):.4f}")

# 8) 학습 완료 후 한 번만 샘플링 & 시각화
model.eval()
with torch.no_grad():
    samples = sample_loop(16)
grid = utils.make_grid(samples, nrow=4)

plt.figure(figsize=(4,4))
plt.axis('off')
plt.imshow(grid.cpu().permute(1,2,0), interpolation='nearest')
plt.savefig('ddpm_samples.png', bbox_inches='tight')
plt.close()

# GPU 캐시 비우기
torch.cuda.empty_cache()

print("샘플 이미지를 ddpm_samples.png로 저장했습니다.")


CUDA available: True
Device: NVIDIA GeForce RTX 3060
Epoch 1/20 — Loss: 0.2056
Epoch 2/20 — Loss: 0.0979
Epoch 3/20 — Loss: 0.0819
Epoch 4/20 — Loss: 0.0750
Epoch 5/20 — Loss: 0.0700
Epoch 6/20 — Loss: 0.0664
Epoch 7/20 — Loss: 0.0650
Epoch 8/20 — Loss: 0.0633
Epoch 9/20 — Loss: 0.0617
Epoch 10/20 — Loss: 0.0608
Epoch 11/20 — Loss: 0.0596
Epoch 12/20 — Loss: 0.0585
Epoch 13/20 — Loss: 0.0579
Epoch 14/20 — Loss: 0.0573
Epoch 15/20 — Loss: 0.0563
Epoch 16/20 — Loss: 0.0562
Epoch 17/20 — Loss: 0.0558
Epoch 18/20 — Loss: 0.0545
Epoch 19/20 — Loss: 0.0542
Epoch 20/20 — Loss: 0.0543


: 